# PythonLab3 Causality of financial time series

The following notebook contains a python implementation of the original RLab on Causality of financial time series by Argimiro Arratia @2023




In [4]:
pip install tensorflow

  Using cached tensorflow-2.16.2-cp39-cp39-macosx_10_15_x86_64.whl (259.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 7.6 MB/s eta 0:00:0000:0100:01
  Using cached opt_einsum-3.4.0-py3-none-any.whl (71 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 6.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached google_pasta-0.2.0-py3-none-any.whl (57 kB)
  Using cached tensorflow_io_gcs_filesystem-0.37.1-cp39-cp39-macosx_10_14_x86_64.whl (2.5 MB)
  Using cached libclang-18.1.1-py2.py3-none-macosx_10_9_x86_64.whl (26.5 MB)
  Using cached tensorboard-2.16.2-py3-none-any.whl (5.5 MB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.8/394.8 kB 6.2 MB/s eta 0:00:0000:01
  Using cached ml_dtypes-0.3.2-cp39-cp39-macosx_10_9_universal2.whl (389 kB)
  Using cached tensorboard_data_server

##2. Libraries:
We will use the following libraries for the study along with their purpose:

In [ ]:
# pandas for data management
import pandas as pd

# numpy for computation
import numpy as np

# pyplot for plotting
import matplotlib.pyplot as plt

# statsmodels for statistical models
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import grangercausalitytests

# keras for deep neural networks #Needed to implement a Terasvirta test
from keras.models import Sequential
from keras.layers import Dense

# scipy.stats for distributions
from scipy.stats import chi2

ModuleNotFoundError: No module named 'tensorflow'

## 3. Data Gathering and Processing:
We begin by loading our DataFrame from the GoyalMonthly2005 csv source containing the S&P500 monthly records up to 2005. We filter our the values previous to 1927 as some fields in the dataset were not registered previous to that time.

In [ ]:
sp500 = pd.read_csv('GoyalMonthly2005.csv', sep=',')
mt = sp500.loc[sp500['Date'] >= '1927-01-01']

Once we have loaded our datasource we compute the log equity premium (GSPCep) and the log returns (logret) of the SP500.

In [ ]:
# Compute log equity premium (GSPCep), log returns of SP500 (logret)
logret = np.log(mt['Index']).diff()
IndexDiv = mt['Index'] + mt['D12']
logretdiv = np.log(IndexDiv).diff()
logRfree = np.log(mt['Rfree'] + 1)
GSPCep = logretdiv - logRfree
GSPCep.name = "GSPCep"

# Keep the "Date" field in the series
GSPCep = pd.Series(data=GSPCep.values, index=mt['Date'], name="GSPCep")

We continue by computing a bunch of financial indicators separately into pandas series with date index.

In [ ]:
# dividend-price ratio (dp)
dp = np.log(mt['D12']) - np.log(mt['Index'])
dp.name = "dp"
dp = pd.Series(dp.values, index=mt['Date'])

# dividend-payout ratio (de)
de = np.log(mt['D12']) - np.log(mt['E12'])
de.name = "de"
de = pd.Series(de.values, index=mt['Date'])

# earnings to price
ep = np.log(mt['E12']) - np.log(mt['Index'])
ep.name = "ep"
ep = pd.Series(ep.values, index=mt['Date'])

## dividend yield
dy = np.log(mt['D12']) - np.log(mt['Index'].shift(1))
dy.name = "dy"
dy = pd.Series(dy.values, index=mt['Date'])

# Default yield spread (dfy)= BAA-AAA rated corporate bond yields:
dfy = mt['BAA'] - mt['AAA']
dfy.name = "dfy"
dfy = pd.Series(dfy.values, index=mt['Date'])

# stock variance (svar)
svar = mt['svar']
svar.name = "svar"
svar = pd.Series(svar.values, index=mt['Date'])

# Book-to-Market (b/m)
bm = mt['b/m']
bm.name = "bm"
bm = pd.Series(bm.values, index=mt['Date'])

# net equity expansion (ntis, start 1926)
ntis = mt['ntis']
ntis.name = "ntis"
ntis = pd.Series(ntis.values, index=mt['Date'])

# inflation (infl)
infl = mt['infl']
infl.name = "infl"
infl = pd.Series(infl.values, index=mt['Date'])

# Treasury Bill rates (tbl, 1920)
tbl = mt['tbl']
tbl.name = "tbl"
tbl = pd.Series(tbl.values, index=mt['Date'])

## 4. Causality Analysis:

We define our target (GPSCep) and predictors (dp, svar) for the study. The lags are not included here as they will be handled by our fitting algorithms.

In [ ]:
Z = pd.concat([GSPCep, dp, svar], axis=1)
Z.columns = ['GSPCep', 'dp', 'svar']

We define some time intervals and compute a granger test analysing cauality of dp over target and svar over target at each epoch. The epoch selection is arbitrary.  
To be more precise:
* plot target and vars and consider periods of common smooth behaviour between two break points
* keep in mind: data is monthly so need 4-5 years for enough data for tests

In [ ]:
# List of date intervals
intervals = [["1927-01-01", "1932-12-01"],
             ["1933-01-01", "1970-12-01"],
             ["1971-01-01", "1997-12-01"],
             ["1998-01-01", "2005-12-01"]]
# Loop for granger test in specified date intervals
causality_tests = []
for interval in intervals:
    Zp = Z[(Z.index >= interval[0]) & (Z.index <= interval[1])].dropna()
    # select the maximum significant lag p
    model = VAR(Zp)
    p = model.select_order(maxlags=5).selected_orders["aic"]
    # Perform Granger causality test for each lag
    causality_test_results = []
    print('\n\n',interval[0],' , ',interval[1],'\n p= ',p,'\n')
    for h in range(1, p+1):
        # Whether the time series in the second column Granger causes the time series in the first column for all lags up to h
        result = grangercausalitytests(Zp[["GSPCep", "dp"]], h)
        causality_test_results.append(result)
    causality_tests.append(causality_test_results)

/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)




 1927-01-01  ,  1932-12-01 
 p=  3 


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.1631  , p=0.6876  , df_denom=67, df_num=1
ssr based chi2 test:   chi2=0.1704  , p=0.6797  , df=1
likelihood ratio test: chi2=0.1702  , p=0.6799  , df=1
parameter F test:         F=0.1631  , p=0.6876  , df_denom=67, df_num=1

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.1631  , p=0.6876  , df_denom=67, df_num=1
ssr based chi2 test:   chi2=0.1704  , p=0.6797  , df=1
likelihood ratio test: chi2=0.1702  , p=0.6799  , df=1
parameter F test:         F=0.1631  , p=0.6876  , df_denom=67, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=0.4994  , p=0.6092  , df_denom=64, df_num=2
ssr based chi2 test:   chi2=1.0768  , p=0.5837  , df=2
likelihood ratio test: chi2=1.0685  , p=0.5861  , df=2
parameter F test:         F=0.4994  , p=0.6092  , df_denom=64, df_num=2

Granger Causality
number of lags (no zero) 1
ssr based F tes

/usr/local/lib/python3.10/dist-packages/statsmodels/tsa/base/tsa_model.py:471: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


##5. (Extra feature analysis) Correlation:
We compute the correlation and distance correlation tests among predictors to check for independence between features.

wait for changing

In [ ]:
MP = pd.concat([ep, dp, de, dy, dfy, bm, svar, ntis, infl, tbl], axis=1)
interval = ['1971-01-01', '1997-12-01']
correlation = np.corrcoef(MP[(MP.index >= interval[0]) & (MP.index <= interval[1])].dropna())

distance_correlation = np.corrcoef(ep[(ep.index >= interval[0]) & (ep.index <= interval[1])], dp[(dp.index >= interval[0]) & (dp.index <= interval[1])])
distance_correlation = np.corrcoef(ep[(ep.index >= interval[0]) & (ep.index <= interval[1])], de[(de.index >= interval[0]) & (de.index <= interval[1])])
distance_correlation = np.corrcoef(ep[(ep.index >= interval[0]) & (ep.index <= interval[1])], svar[(svar.index >= interval[0]) & (svar.index <= interval[1])])

In [ ]:
correlation ##view correlation

array([[1.        , 0.99998068, 0.99999339, ..., 0.99273486, 0.99331931,
        0.99344699],
       [0.99998068, 1.        , 0.99998786, ..., 0.99316145, 0.99366986,
        0.99381102],
       [0.99999339, 0.99998786, 1.        , ..., 0.99290894, 0.99346183,
        0.99358488],
       ...,
       [0.99273486, 0.99316145, 0.99290894, ..., 1.        , 0.99991005,
        0.99992113],
       [0.99331931, 0.99366986, 0.99346183, ..., 0.99991005, 1.        ,
        0.99998842],
       [0.99344699, 0.99381102, 0.99358488, ..., 0.99992113, 0.99998842,
        1.        ]])

We additionally perform a terasvista test where we test neglected nonlinearity. The null is the hypotheses of linearity in "mean". Test for existence of linearity relation (if rejected then possibly nonlinear) between one variable and target. We do so by fitting a neural network (non-linear model) and performing a chisq test over the residuals of that model. The following function provides a simple implementation using keras for the neural network and scipy stats for the chisq distribution.

Justification behind using non-linear models

In [ ]:
def terasvirta_test(x, y):
  model = Sequential()
  input_dim = 1 if len(x.shape) == 1 else x.shape[1]
  model.add(Dense(2, activation='relu', input_dim=input_dim))
  model.add(Dense(1, activation='relu'))
  model.compile(optimizer='adam', loss='mean_squared_error')
  model.fit(x, y, epochs=50, verbose=False)

  linear_model = Sequential()
  linear_model.add(Dense(1, activation='linear'))
  linear_model.compile(loss='mse', optimizer='sgd')
  linear_model.fit(x, y, epochs=50, verbose=False)
  pred_orig = linear_model.predict(x)
  resid_orig = y.values - pred_orig
  pred_nn = model.predict(x)
  pred_nn = pred_nn.reshape(-1)
  resid_nn = y.values - pred_nn
  test_stat = np.mean(resid_orig**2 - resid_nn**2)
  crit_val = chi2.ppf(0.95, 2)
  if test_stat > crit_val:
      print("The null hypothesis of linearity is rejected")
  else:
    print("The null hypothesis of linearity is not rejected")

In [ ]:
terasvirta_test(ep[(ep.index >= interval[0]) & (ep.index <= interval[1])], GSPCep[(GSPCep.index >= interval[0]) & (GSPCep.index <= interval[1])])

11/11 [==============================] - 0s 7ms/step
The null hypothesis of linearity is not rejected


In [ ]:
terasvirta_test(MP[(MP.index >= interval[0]) & (MP.index <= interval[1])], GSPCep[(GSPCep.index >= interval[0]) & (GSPCep.index <= interval[1])].dropna())

11/11 [==============================] - 0s 1ms/step
The null hypothesis of linearity is not rejected
